# SegmentViewer GL — WebGL2 demo

**Performance characteristics:**
- Segment track: GPU instanced rendering, 1 draw call/group, zero CPU on pan/zoom
- Heatmap track: R8 texture, handles 10 k+ individuals at 60 fps
- Pan/zoom: uniform-only updates — no buffer uploads during interaction
- Culling: sorted attribute-pointer offset skips non-visible instances

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from collections import OrderedDict
from geneinfo.chrom_tracks import SegmentViewer
from vscodenb import set_vscode_theme
set_vscode_theme()


rng = np.random.default_rng(42)

theme = {                                                               
    # 'bg':           '#ffffff', 
    'bg':           '#FAF9F6', 
    'fg':           '#1a1a2e',                                                 
    'panel':        '#FAF9F6',                            
    'border':       '#FAF9F6',                                                 
    'input_bg':     '#ffffff',          
    'input_fg':     '#1a1a2e',                                                 
    'input_border': '#c0c0cc',                                                 
    'accent':       '#3355dd',          
    'muted':        '#888899',                                                 
    'label':        '#555566',                                                 
}

In [2]:
from geneinfo.coords import gene_coords_region, chromosome_lengths

chrom_names, chrom_lengths = zip(*chromosome_lengths(assembly='hg38'))

genes_by_chrom = {}
for chrom in chrom_names:
    genes_by_chrom[chrom] = gene_coords_region(chrom, assembly='hg38')
    print(f'{chrom}:\t{len(genes_by_chrom[chrom]):,} genes')

chrom_sizes = dict(zip(chrom_names, chrom_lengths))

chr1:	4,110 genes
chr2:	3,008 genes
chr3:	2,364 genes
chr4:	1,902 genes
chr5:	2,084 genes
chr6:	2,291 genes
chr7:	2,097 genes
chr8:	1,757 genes
chr9:	1,793 genes
chr10:	1,774 genes
chr11:	2,335 genes
chr12:	2,099 genes
chr13:	1,060 genes
chr14:	1,372 genes
chr15:	1,469 genes
chr16:	1,648 genes
chr17:	2,117 genes
chr18:	801 genes
chr19:	2,153 genes
chr20:	1,142 genes
chr21:	628 genes
chr22:	892 genes
chrX:	1,447 genes
chrY:	210 genes


## 1 · Archaic introgression segments (real data from v2/)

In [3]:
# Load metadata
meta = pq.ParquetFile('/Users/kmt/v2/dims/metadata.parquet').read().to_pandas()

# subset
subset_regs = ['EUR', 'EAS']
subset_meta = meta[meta['reg'].isin(subset_regs)].groupby('reg').head(100)

# # all
# subset_meta = meta
# subset_meta.head()

In [4]:
subset_meta.ind.unique().size

np.int64(200)

In [5]:
# Read fragment parquet files (one per individual)
frames = []
for ind in subset_meta['ind']:
    fp = Path(f'/Users/kmt/v2/fragments/phase_state=unphased/ind={ind}/0.parquet')
    if fp.exists():
        frames.append(pq.ParquetFile(fp).read().to_pandas())

segs = pd.concat(frames, ignore_index=True)
segs['chrom'] = 'chr' + segs['chrom'].astype(str)
#segs = segs[segs['chrom'] == 'chr1']
segs = segs.merge(meta[['ind', 'reg', 'pop', 'sex']], on='ind', how='left')

#chrom_sizes = {'chr1': 248_956_422}

print(f'{len(segs):,} segments from {segs["ind"].nunique()} individuals')
print(f'Mean length: {(segs["end"] - segs["start"]).mean():,.0f} bp')
print(f'Regions: {dict(segs.groupby("reg")["ind"].nunique())}')
print(f'Ancestries: {sorted(segs["ancestry"].unique())}')

339,106 segments from 200 individuals
Mean length: 65,971 bp
Regions: {'EAS': np.int64(100), 'EUR': np.int64(100)}
Ancestries: ['Altai', 'AmbigNean', 'Ambiguous', 'Chagyrskaya', 'Denisova', 'Vindija', 'nonDAVC']


In [6]:
segs.head()

,chrom,start,end,length,hap,mean_prob,called_sequence,mutationrate,snps,admixpopvariants,...,min_dist_anc,min_dist_value,z_test_anc,z_test_dist,z_test_pval,ind,phase_state,reg,pop,sex
0,chr1,2970000,3056000,86000,1,0.95041,76887,1.364,44,34,...,Chagyrskaya,0.001481,NEA,"4.5498e-03,1.4812e-03,2.3568e-03","3.22e-23,5.00e-01,2.39e-04",AB02,unphased,EAS,AytaMagbukon,F
1,chr1,3723000,3759000,36000,1,0.93465,28544,1.270,16,10,...,Altai,0.000762,NEA,"3.7741e-03,7.6192e-04,6.0505e-03","1.46e-11,5.00e-01,9.58e-22",AB02,unphased,EAS,AytaMagbukon,F
2,chr1,3967000,4027000,60000,1,0.86498,52449,1.252,19,14,...,Vindija,0.001876,NEA,"3.2574e-03,1.8761e-03,3.1197e-03","4.31e-05,5.00e-01,1.95e-04",AB02,unphased,EAS,AytaMagbukon,F
3,chr1,4101000,4119000,18000,1,0.92891,15171,1.230,11,7,...,Chagyrskaya,0.000526,NEA,"2.0667e-03,5.2612e-04,4.4505e-03","3.27e-04,5.00e-01,1.23e-10",AB02,unphased,EAS,AytaMagbukon,F
4,chr1,4446000,4472000,26000,1,0.93772,23031,1.230,15,1,...,Denisova,0.003179,AMB,"4.5570e-03,3.3623e-03,3.1794e-03","1.77e-02,3.82e-01,5.00e-01",AB02,unphased,EAS,AytaMagbukon,F


In [7]:
viewer = (
    SegmentViewer(chrom_sizes)
    .add_segment_track(
        segs, 'Introgression by region',
        group_by='reg',
        individual_col='ind',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc', 'EAS': '#33aa77'},
        height=90,
        density_windows=(512, 2048, 8192),
    )
    .add_segment_track(
        segs, 'Introgression by ancestry',
        group_by='ancestry',
        individual_col='ind',
        height=90,
        density_windows=(512, 2048, 8192),
    )
)
viewer.theme = theme                                      
viewer.zoom_speed = 1.015
viewer

## 2 · Heatmap + segment tracks

In [8]:
# Prepare heatmap input: needs chrom, start, end, sample (individual), pop (group)
hm_df = segs[['chrom', 'start', 'end', 'ind', 'reg']].rename(
    columns={'ind': 'sample', 'reg': 'pop'}
)

In [9]:
viewer2 = (
    SegmentViewer(chrom_sizes)
    .add_segment_track(
        segs, 'Introgression by region',
        group_by='reg',
        individual_col='ind',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc', 'EAS': '#33aa77'},
        height=90,
        density_windows=(512, 2048, 8192),
    )
    .add_heatmap_track(
        hm_df, 'Individual haplotypes',
        individual_col='sample',
        group_col='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc', 'EAS': '#33aa77'},
#        height=250,
        windows=1500,
    )
)

In [10]:
viewer2.theme = theme                                      
viewer2.zoom_speed = 1.015
viewer2

## 3 · Gene annotation track

In [11]:
viewer2.add_gene_track(genes_by_chrom, name='Genes', height=44, collapse=True)

## 4 · New track types: scatter, line, fill-between, histogram

In [12]:
# Generate synthetic quantitative data for chr1
quant_rows = []
for chrom, csz in chrom_sizes.items():
    n_points = 2000
    positions = np.sort(rng.integers(0, csz, n_points))
    fst = np.abs(rng.normal(0.05, 0.08, n_points)).clip(0, 1)
    rate = np.cumsum(rng.normal(0, 0.3, n_points))
    rate = (rate - rate.min()) / (rate.max() - rate.min()) * 5 + 0.5
    mid = np.cumsum(rng.normal(0, 0.15, n_points))
    ci_lo = mid - rng.exponential(0.3, n_points)
    ci_hi = mid + rng.exponential(0.3, n_points)
    depth = rng.poisson(30, n_points).astype(float)

    for pop in ['AFR', 'EUR']:
        noise = rng.normal(0, 0.02, n_points)
        for i in range(n_points):
            quant_rows.append({
                'chrom': chrom, 'pos': int(positions[i]),
                'fst': float(fst[i] + noise[i]),
                'rate': float(rate[i] + noise[i] * 10),
                'ci_lo': float(ci_lo[i]), 'ci_hi': float(ci_hi[i]),
                'depth': float(depth[i] + rng.normal(0, 3)),
                'pop': pop,
            })

qdf = pd.DataFrame(quant_rows)
print(f'{len(qdf):,} data points')

96,000 data points


In [13]:
viewer3 = (
    SegmentViewer(chrom_sizes)
    .add_scatter_track(
        qdf, 'Fst',
        x='pos', y='fst',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
        point_size=2,
    )
    .add_line_track(
        qdf, 'Recombination rate',
        x='pos', y='rate',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
    )
    .add_fill_track(
        qdf, 'Confidence interval (lo/hi)',
        x='pos', y='ci_lo',
        group_by='pop',
        color_pos="#ee5555",
        color_neg="#5567ee",
        height=70,
        baseline=0.0,
    )
    .add_fill_track(
        qdf, 'Fst (single y, fill to baseline)',
        x='pos', y='fst',
        group_by='pop',
        color_pos='#4488cc',
        color_neg='#ee5566',
        height=70,
    )
    .add_histogram_track(
        qdf, 'Sequencing depth',
        x='pos', y='depth',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
    )
    .add_gene_track(genes_by_chrom, name='Genes', height=44, collapse=True)
)
viewer3.zoom_speed = 1.015
viewer3.theme = theme                                      
viewer3

In [14]:
# Jump to a 5 Mb window around a position of interest
viewer2.zoom_to('chr1', center=100_000_000, window=5_000_000)

In [15]:
# Read current viewport from Python (updated on mouseup / wheel settle)
viewer2.viewport

{'chrom': 'chr1', 'start': 97500000, 'end': 102500000}

## 5 · Auto-sizing track height

When `height` is omitted, the track auto-sizes to `max(90, total_rows)` so every
individual gets at least one pixel of vertical space.

In [16]:
# With 200 individuals the auto-height expands the track beyond the 90px minimum.
# Compare: the segment track (no individual_col) stays at 90px,
# while the heatmap (200 individuals) grows to 200px.
viewer4 = (
    SegmentViewer(chrom_sizes)
    .add_segment_track(
        segs, 'Segments (auto height)',
        group_by='reg',
        color_map={'EUR': '#4488cc', 'EAS': '#33aa77'},
    )
    .add_heatmap_track(
        hm_df, 'Heatmap (auto height)',
        individual_col='sample',
        group_col='pop',
        color_map={'EUR': '#4488cc', 'EAS': '#33aa77'},
        windows=1500,
    )
    .add_gene_track(genes_by_chrom, name='Genes', height=44, collapse=True)
)
viewer4.theme = theme
viewer4.zoom_speed = 1.015
viewer4

## 6 · Proportional group bands

Groups are sized proportionally to their row count. Here EUR has 100 individuals
and EAS has 100, so they get equal space. With unequal counts the larger group
would get proportionally more vertical space.

In [17]:
# Create an unbalanced subset: 20 EUR vs 80 EAS individuals
eur_inds = meta[meta['reg'] == 'EUR'].head(20)['ind'].tolist()
eas_inds = meta[meta['reg'] == 'EAS'].head(80)['ind'].tolist()
unbal_inds = set(eur_inds + eas_inds)
unbal_segs = segs[segs['ind'].isin(unbal_inds)]
unbal_hm = unbal_segs[['chrom', 'start', 'end', 'ind', 'reg']].rename(
    columns={'ind': 'sample', 'reg': 'pop'}
)

viewer5 = (
    SegmentViewer(chrom_sizes)
    .add_segment_track(
        unbal_segs, 'Segments (20 EUR vs 80 EAS)',
        group_by='reg',
        individual_col='ind',
        color_map={'EUR': '#4488cc', 'EAS': '#33aa77'},
    )
    .add_heatmap_track(
        unbal_hm, 'Heatmap (20 EUR vs 80 EAS)',
        individual_col='sample',
        group_col='pop',
        color_map={'EUR': '#4488cc', 'EAS': '#33aa77'},
        windows=1500,
    )
    .add_gene_track(genes_by_chrom, name='Genes', height=44, collapse=True)
)
viewer5.theme = theme
viewer5.zoom_speed = 1.015
viewer5

## 7 · Cursor tooltips

Hover over any track to see context-sensitive information in the tooltip:

- **Gene track**: gene name and strand direction
- **Scatter / line track**: nearest y-value (per group if grouped)
- **Fill track**: nearest lo-hi range
- **Histogram track**: bin value under cursor
- **Segment track**: group name at cursor position
- **Heatmap track**: group name and individual index

The genome coordinate is always shown on the first line.

In [18]:
# This viewer combines all track types — hover over each to see the tooltip.
viewer6 = (
    SegmentViewer(chrom_sizes)
    .add_segment_track(
        segs, 'Introgression',
        group_by='reg',
        individual_col='ind',
        color_map={'EUR': '#4488cc', 'EAS': '#33aa77'},
        density_windows=(512, 2048, 8192),
    )
    .add_heatmap_track(
        hm_df, 'Haplotypes',
        individual_col='sample',
        group_col='pop',
        color_map={'EUR': '#4488cc', 'EAS': '#33aa77'},
        windows=1500,
    )
    .add_scatter_track(
        qdf, 'Fst',
        x='pos', y='fst',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
        point_size=2,
    )
    .add_line_track(
        qdf, 'Recombination rate',
        x='pos', y='rate',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
    )
    .add_fill_track(
        qdf, 'Confidence interval',
        x='pos', y_lo='ci_lo', y_hi='ci_hi',
        group_by='pop',
        color_pos='#33aa77',
        color_neg='#ee5566',
        height=70,
        baseline=0.0,
    )
    .add_histogram_track(
        qdf, 'Sequencing depth',
        x='pos', y='depth',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
    )
    .add_gene_track(genes_by_chrom, name='Genes', height=44, collapse=True)
)
viewer6.theme = theme
viewer6.zoom_speed = 1.015
viewer6

## 8 · Custom tooltip format strings

Use `tip_fmt` to control what each track contributes to the tooltip.
Format placeholders use Python-style `{key}` or `{key:.Nf}` syntax.

Available keys per track type:
- **Gene**: `name`, `strand`, `start`, `end`
- **Scatter / Line**: `group`, `value`, `x`
- **Fill**: `group`, `lo`, `hi`, `x`
- **Histogram**: `group`, `value`, `x`
- **Segment**: `group`
- **Heatmap**: `group`, `individual`, `nInd`

If a format string references an invalid key, the tooltip shows the
available keys as a hint.

In [20]:
viewer7 = (
    SegmentViewer(chrom_sizes)
    .add_scatter_track(
        qdf, 'Fst',
        x='pos', y='fst',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
        point_size=2,
        tip_fmt="{group}: {value:.3f}",
    )
    .add_line_track(
        qdf, 'Rec. rate',
        x='pos', y='rate',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
        tip_fmt="{group}: {value:.2f} cM/Mb",
    )
    .add_histogram_track(
        qdf, 'Depth',
        x='pos', y='depth',
        group_by='pop',
        color_map={'AFR': '#ee5566', 'EUR': '#4488cc'},
        height=60,
        # tip_fmt="{group}: {value:.0f}x",
        tip_fmt=False,
    )
    .add_gene_track(
        genes_by_chrom, name='Genes', height=44, collapse=True,
        tip_fmt="{name}",
    )
)
viewer7.theme = theme
viewer7.zoom_speed = 1.015
viewer7